# Build a sentiment analysis model using either Hugging Face's DistiBERT or TensorFlow's IMDB Movie Reviews Dataset

In [3]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis")

texts = ["I Love this movie", "This movie was terrible"]
results = sentiment_pipeline(texts)

for text, result in zip(texts, results):
    print(f"Text: {text} | Sentiment: {result['label']}, Score: {result['score']}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I Love this movie | Sentiment: POSITIVE, Score: 0.9998766183853149
Text: This movie was terrible | Sentiment: NEGATIVE, Score: 0.9996950626373291


In [10]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

(train_data, test_data), info = tfds.load(
    'imdb_reviews',
    split=['train', 'test'],
    with_info=True,
    as_supervised=True
)

train_texts = []
train_labels = []

for text, label in tfds.as_numpy(train_data):
    train_texts.append(text.decode('utf-8'))
    train_labels.append(label)

train_labels = np.array(train_labels)

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)

train_padded = pad_sequences(
    train_sequences,
    maxlen=120,
    padding='post',
    truncating='post'
)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 16, input_length=120),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_padded,
    train_labels,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5


C:\Users\bandi\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


625/625 ━━━━━━━━━━━━━━━━━━━━ 23s 33ms/step - accuracy: 0.5796 - loss: 0.6696 - val_accuracy: 0.6018 - val_loss: 0.6576
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.6159 - loss: 0.6481 - val_accuracy: 0.5862 - val_loss: 0.6759
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.7066 - loss: 0.5821 - val_accuracy: 0.7774 - val_loss: 0.5102
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.8132 - loss: 0.4314 - val_accuracy: 0.7910 - val_loss: 0.4795
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.8592 - loss: 0.3480 - val_accuracy: 0.8044 - val_loss: 0.4284
